In [1]:
import os
import numpy as np
import pandas as pd

# ---- Team-agreed rules (change here if the team updates them) ----
MIN_AGE = 20
EXCLUDE_PREGNANT = True
DIAGNOSED_CODES = [1, 7, 9]   # DIQ010: 1 = yes, 7 = refused, 9 = don't know (3 = borderline is kept)
PRIMARY_CUT = 6.5
SECONDARY_CUT = 5.7

DATA = "../data" if os.path.exists("../data") else "data"

def load(name):
    for root, _, files in os.walk(DATA):
        for f in files:
            if f.upper().startswith(name) and f.upper().endswith(".XPT"):
                df = pd.read_sas(os.path.join(root, f))
                num = df.select_dtypes("number").columns
                df[num] = df[num].mask(df[num].abs() < 1e-10, 0)
                print("Loaded", os.path.join(root, f), df.shape)
                return df
    print("NOT FOUND:", name)

demo, cbc, ghb, diq = load("DEMO_L"), load("CBC_L"), load("GHB_L"), load("DIQ_L")

df = (demo.merge(cbc, on="SEQN")
          .merge(ghb.drop(columns="WTPH2YR"), on="SEQN")
          .merge(diq[["SEQN", "DIQ010"]], on="SEQN", how="left"))

flow = [("Has CBC and HbA1c records", len(df))]
d = df[df["RIDAGEYR"] >= MIN_AGE];                              flow.append((f"Age {MIN_AGE}+", len(d)))
if EXCLUDE_PREGNANT:
    d = d[d["RIDEXPRG"] != 1];                                  flow.append(("Not pregnant", len(d)))
d = d[d["DIQ010"].notna() & ~d["DIQ010"].isin(DIAGNOSED_CODES)]; flow.append(("No prior diabetes diagnosis", len(d)))
d = d[(d["WTPH2YR"] > 0) & d["LBXGH"].notna()];                 flow.append(("Valid weight and HbA1c", len(d)))
coh = d.copy()

display(pd.DataFrame(flow, columns=["Step", "Participants"]))

rows = []
for label, cut in [("Primary", PRIMARY_CUT), ("Secondary", SECONDARY_CUT)]:
    y = (coh["LBXGH"] >= cut).astype(int)
    rows.append({"Outcome": f"{label}: HbA1c >= {cut}%",
                 "Cases (n)": int(y.sum()),
                 "Unweighted %": round(100 * y.mean(), 1),
                 "Weighted %": round(100 * np.average(y, weights=coh["WTPH2YR"]), 1)})
display(pd.DataFrame(rows))

Loaded ../data\raw\2021_2023\DEMO_L.xpt (11933, 27)
Loaded ../data\raw\2021_2023\CBC_L.xpt (8727, 23)
Loaded ../data\raw\2021_2023\GHB_L.xpt (7199, 3)
Loaded ../data\raw\2021_2023\DIQ_L.xpt (11744, 9)


,Step,Participants
0,Has CBC and HbA1c records,7199
1,Age 20+,6064
2,Not pregnant,6023
3,No prior diabetes diagnosis,5155
4,Valid weight and HbA1c,4911


,Outcome,Cases (n),Unweighted %,Weighted %
0,Primary: HbA1c >= 6.5%,152,3.1,2.6
1,Secondary: HbA1c >= 5.7%,1541,31.4,27.3


In [2]:
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import RepeatedStratifiedKFold
from sklearn.metrics import roc_auc_score

bmx, crp, fer = load("BMX_L"), load("HSCRP_L"), load("FERTIN_L")
data = (coh.merge(bmx[["SEQN", "BMXBMI", "BMXWAIST"]], on="SEQN", how="left")
           .merge(crp[["SEQN", "LBXHSCRP"]], on="SEQN", how="left")
           .merge(fer[["SEQN", "LBXFER"]], on="SEQN", how="left"))
data["male"] = (data["RIAGENDR"] == 1).astype(int)
data["log_crp"] = np.log1p(data["LBXHSCRP"])

# Team's 15 CBC features (same as cbc_vars in 02_baseline); no absolute counts, no NRBC
CBC = ["LBXWBCSI", "LBXRBCSI", "LBXHGB", "LBXHCT", "LBXMCVSI", "LBXMCHSI", "LBXMC", "LBXRDW",
       "LBXPLTSI", "LBXMPSI", "LBXLYPCT", "LBXNEPCT", "LBXMOPCT", "LBXEOPCT", "LBXBAPCT"]
RED = ["LBXRBCSI", "LBXHGB", "LBXHCT", "LBXMCVSI", "LBXMCHSI", "LBXMC", "LBXRDW"]
NO_RED = [c for c in CBC if c not in RED]
BASE = ["RIDAGEYR", "male", "BMXBMI"]

model = make_pipeline(SimpleImputer(strategy="median"), StandardScaler(),
                      LogisticRegression(max_iter=2000))
cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=5, random_state=42)

def evaluate(d, sets, ref, cut=PRIMARY_CUT):
    d = d.reset_index(drop=True)
    y = (d["LBXGH"] >= cut).astype(int).values
    w = d["WTPH2YR"].values
    splits = list(cv.split(d, y))
    scores = {}
    for name, cols in sets.items():
        s = []
        for tr, te in splits:
            model.fit(d[cols].iloc[tr], y[tr])
            p = model.predict_proba(d[cols].iloc[te])[:, 1]
            s.append(roc_auc_score(y[te], p, sample_weight=w[te]))   # survey-weighted AUROC
        scores[name] = np.array(s)
    print(f"n = {len(d)} | cases (HbA1c >= {cut}%) = {int(y.sum())}")
    return pd.DataFrame([{"Model": n, "Features": len(sets[n]),
                          "Weighted AUROC": round(s.mean(), 3), "SD": round(s.std(), 3),
                          f"Change vs {ref}": round((s - scores[ref]).mean(), 3)}
                         for n, s in scores.items()])
print("Ready")

Loaded ../data\raw\2021_2023\BMX_L.xpt (8860, 22)
Loaded ../data\raw\2021_2023\laboratory\HSCRP_L.xpt (8727, 4)
Loaded ../data\raw\2021_2023\laboratory\FERTIN_L.xpt (2564, 4)
Ready


In [3]:
sets = {
    "A: age + sex": ["RIDAGEYR", "male"],
    "A2: age + sex + BMI": BASE,
    "B: A2 + CBC": BASE + CBC,
    "C: A2 + CBC without red-cell": BASE + NO_RED,
    "D: B + waist": BASE + CBC + ["BMXWAIST"],
    "E: B + hs-CRP": BASE + CBC + ["log_crp"],
}
evaluate(data, sets, ref="A2: age + sex + BMI")

n = 4911 | cases (HbA1c >= 6.5%) = 152


,Model,Features,Weighted AUROC,SD,Change vs A2: age + sex + BMI
0,A: age + sex,2,0.686,0.060,-0.098
1,A2: age + sex + BMI,3,0.784,0.042,0.000
2,B: A2 + CBC,18,0.805,0.045,0.021
3,C: A2 + CBC without red-cell,11,0.800,0.041,0.016
4,D: B + waist,19,0.807,0.044,0.023
5,E: B + hs-CRP,19,0.822,0.050,0.038


In [4]:
hb = data["LBXHGB"]
anemic = ((data["male"] == 1) & (hb < 13)) | ((data["male"] == 0) & (hb < 12))
na = data[hb.notna() & ~anemic]
print("Excluded:", int(anemic.sum()), "anemic,", int(hb.isna().sum()), "missing hemoglobin")
display(evaluate(na, {"A2: age + sex + BMI": BASE, "B: A2 + CBC": BASE + CBC},
                 ref="A2: age + sex + BMI"))

sub = data[data["LBXFER"].notna()]
print("Iron subgroup:", len(sub), "people | cases at >= 6.5%:",
      int((sub["LBXGH"] >= PRIMARY_CUT).sum()))

Excluded: 451 anemic, 18 missing hemoglobin
n = 4442 | cases (HbA1c >= 6.5%) = 136


,Model,Features,Weighted AUROC,SD,Change vs A2: age + sex + BMI
0,A2: age + sex + BMI,3,0.789,0.049,0.000
1,B: A2 + CBC,18,0.804,0.050,0.014


Iron subgroup: 1130 people | cases at >= 6.5%: 19


In [5]:
evaluate(data, sets, ref="A2: age + sex + BMI", cut=SECONDARY_CUT)

n = 4911 | cases (HbA1c >= 5.7%) = 1541


,Model,Features,Weighted AUROC,SD,Change vs A2: age + sex + BMI
0,A: age + sex,2,0.705,0.017,-0.032
1,A2: age + sex + BMI,3,0.736,0.015,0.000
2,B: A2 + CBC,18,0.787,0.012,0.051
3,C: A2 + CBC without red-cell,11,0.754,0.016,0.018
4,D: B + waist,19,0.787,0.012,0.050
5,E: B + hs-CRP,19,0.792,0.012,0.056
